# Notebook 04 – Teste final com dados de 2026

Este notebook executa a avaliação final do modelo escolhido (arquitetura **64→32**, limiar 0,5) usando o recorte de **2026** como conjunto de teste final, conforme definido no roteiro do desafio.

O objetivo é comparar as classes reais da base de dados com as classes preditas pelo modelo, apresentando métricas (matriz de confusão, precisão, recall, F1-macro) e gráficos comparativos.

Não há novo treino nem novo ajuste de pré-processamento: o modelo e o pré-processador são carregados dos artefatos salvos em `models/best/`.

In [6]:
# Importa pandas para manipulação de tabelas.
import pandas as pd
# Importa numpy para operações numéricas.
import numpy as np
# Importa torch para carregar o modelo e realizar inferência.
import torch
# Importa nn para reconstruir a arquitetura da rede.
import torch.nn as nn
# Importa joblib para carregar o pré-processador serializado.
import joblib
# Importa json para carregar o mapeamento de rótulos.
import json
# Importa Path para montar caminhos de arquivos.
from pathlib import Path
# Importa funções de sklearn para métricas e matriz de confusão.
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
# Importa matplotlib e configura backend Agg (sem interface gráfica, necessário para execução como script).
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# Importa seaborn para heatmap da matriz de confusão.
import seaborn as sns
# Define a raiz do projeto como o diretório de trabalho atual.
RAIZ = Path.cwd().parent
# Define a pasta com os artefatos do modelo escolhido.
DIR_MODELO = RAIZ / 'models' / 'best'
# Define a pasta dos dados brutos.
DIR_RAW = RAIZ / 'data' / 'raw'

In [7]:
# Importa o carregador de dados do projeto.
from src.data.loader import carregar_e_preparar
# Importa a função de consolidação por boletim.
from src.data.splitter import consolidar_boletins
# Importa a função de extração de features temporais.
from src.data.preprocessor import extrair_features_temporais, CATEGORICAS_NOMINAIS
# Carrega os dados brutos de 2026, filtra São Paulo e define o alvo binário.
df_2026 = carregar_e_preparar(ano=2026)
# Exibe o número de linhas carregadas após o filtro.
print(f'Linhas carregadas de 2026: {len(df_2026)}')
# Exibe a contagem de ocorrências por classe (target) em 2026.
print('Contagem por classe (target):')
print(df_2026['target'].value_counts().sort_index())

Linhas carregadas de 2026: 126567
Contagem por classe (target):
target
0    77003
1    49564
Name: count, dtype: int64


In [8]:
# Consolida múltiplas linhas do mesmo BO em uma única linha (mesma lógica do treino).
df_consolidado = consolidar_boletins(df_2026)
# Exibe o número de BOs únicos após a consolidação.
print(f'BOs únicos consolidados de 2026: {len(df_consolidado)}')
# Exibe a contagem por classe após a consolidação.
print('Contagem por classe após consolidação:')
print(df_consolidado['target'].value_counts().sort_index())
# Exibe as primeiras linhas para inspeção visual.
print(df_consolidado.head().to_string())

BOs únicos consolidados de 2026: 112192
Contagem por classe após consolidação:
target
0    72917
1    39275
Name: count, dtype: int64
   NUM_BO  target DESCR_TIPOLOCAL DESCR_SUBTIPOLOCAL              BAIRRO DESCR_PERIODO HORA_OCORRENCIA DATA_OCORRENCIA_BO
0  AA0040       0     Via Pública        Via Pública          BELA VISTA       missing        23:37:00         2025-12-31
1  AA0139       1     Via Pública        Via Pública  JARDIM VILA CARRAO       missing        23:00:00         2025-12-31
2  AA0203       0     Via Pública        Via Pública          BELA VISTA       missing        22:20:00         2025-12-31
3  AA0225       0     Via Pública        Via Pública          BELA VISTA       missing        00:10:00         2026-01-01
4  AA0246       1           Saúde        Laboratório             BUTANTÃ       missing        00:00:00         2026-01-01


In [9]:
# Carrega o pré-processador ajustado no treino (sem novo fit, apenas transform).
preprocessador = joblib.load(DIR_MODELO / 'preprocessor.pkl')
# Extrai as features temporais derivadas (dia_semana, mes, hora) a partir de data e hora.
df_consolidado = extrair_features_temporais(df_consolidado)
# Seleciona apenas as colunas de features necessárias (categóricas + numéricas derivadas).
colunas_necessarias = CATEGORICAS_NOMINAIS + ['dia_semana', 'mes', 'hora']
X_2026 = df_consolidado[colunas_necessarias]
# Aplica a transformação do pré-processador (One-Hot + StandardScaler) sem refit.
X_transformado = preprocessador.transform(X_2026)
# Converte para array denso se necessário.
if hasattr(X_transformado, 'toarray'):
    X_transformado = X_transformado.toarray()
# Guarda o vetor de classes reais (target) para comparação posterior.
y_real = df_consolidado['target'].values
# Exibe as dimensões da matriz de entrada.
print(f'Dimensão da entrada: {X_transformado.shape}')
# Exibe as dimensões do vetor de classes reais.
print(f'Dimensão do alvo real: {y_real.shape}')

Dimensão da entrada: (112192, 1914)
Dimensão do alvo real: (112192,)


In [10]:
# Carrega os metadados do modelo escolhido.
with open(DIR_MODELO / 'metadata.json', encoding='utf-8') as f:
    metadados = json.load(f)
# Define a classe MLP com a mesma estrutura do modelo salvo (layers.N + output_layer).
class MLP(nn.Module):
    def __init__(self, input_size, hidden_sizes, dropout=0.2):
        super().__init__()
        self.layers = nn.ModuleList()
        self.activations = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        tamanhos = [input_size] + list(hidden_sizes)
        for i in range(len(hidden_sizes)):
            self.layers.append(nn.Linear(tamanhos[i], tamanhos[i + 1]))
            self.activations.append(nn.ReLU())
            self.dropouts.append(nn.Dropout(dropout))
        self.output_layer = nn.Linear(tamanhos[-1], 1)
    def forward(self, x):
        for camada, ativacao, dropout in zip(self.layers, self.activations, self.dropouts):
            x = dropout(ativacao(camada(x)))
        return self.output_layer(x)
# Instancia a rede com o número de features e camadas ocultas do modelo salvo.
modelo = MLP(
    input_size=int(metadados['n_features']),
    hidden_sizes=list(metadados['hidden_sizes']),
    dropout=float(metadados.get('dropout', 0.2))
)
# Carrega os pesos treinados (CPU, suficiente para inferência).
modelo.load_state_dict(torch.load(DIR_MODELO / 'mlp_model.pth', map_location='cpu'))
# Coloca a rede em modo de avaliação (desliga o dropout).
modelo.eval()
# Exibe informações do modelo carregado.
print(f'Arquitetura: {metadados["architecture"]}')
print(f'Limiar de decisão: {metadados["threshold"]}')
print(f'Época do melhor modelo: {metadados.get("best_epoch", "N/A")}')

Arquitetura: 64→32
Limiar de decisão: 0.5
Época do melhor modelo: 9


In [11]:
# Converte a matriz transformada em tensor float32.
X_tensor = torch.tensor(X_transformado, dtype=torch.float32)
# Desativa o cálculo de gradientes durante a inferência.
with torch.no_grad():
    # Calcula os logits da rede para todos os BOs de 2026.
    logits = modelo(X_tensor).squeeze(1)
    # Converte os logits em probabilidade de roubo via sigmoide.
    prob_roubo = torch.sigmoid(logits).numpy()
# Aplica o limiar de decisão salvo nos metadados.
limiar = float(metadados['threshold'])
y_pred = (prob_roubo >= limiar).astype(int)
# Cria um DataFrame com os resultados para facilitar a comparação.
df_resultados = pd.DataFrame({
    'target_real': y_real,
    'target_predito': y_pred,
    'prob_roubo': prob_roubo
})
# Mapeia os códigos numéricos para rótulos legíveis.
mapa_labels = {0: 'Furto', 1: 'Roubo'}
df_resultados['classe_real'] = df_resultados['target_real'].map(mapa_labels)
df_resultados['classe_predita'] = df_resultados['target_predito'].map(mapa_labels)
# Exibe as primeiras linhas do resultado.
print(df_resultados.head(10).to_string())
# Exibe a contagem de classes preditas.
print('Contagem de classes preditas:')
print(df_resultados['classe_predita'].value_counts())

   target_real  target_predito  prob_roubo classe_real classe_predita
0            0               1    0.588037       Furto          Roubo
1            1               1    0.940643       Roubo          Roubo
2            0               1    0.534505       Furto          Roubo
3            0               1    0.633563       Furto          Roubo
4            1               0    0.060749       Roubo          Furto
5            0               1    0.652913       Furto          Roubo
6            1               1    0.616437       Roubo          Roubo
7            1               0    0.399028       Roubo          Furto
8            1               0    0.494192       Roubo          Furto
9            1               1    0.621621       Roubo          Roubo
Contagem de classes preditas:
classe_predita
Furto    60426
Roubo    51766
Name: count, dtype: int64


In [12]:
# Importa funções de sklearn para cálculo de métricas.
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
# Calcula a matriz de confusão (linhas = real, colunas = predito).
cm = confusion_matrix(df_resultados['target_real'], df_resultados['target_predito'])
# Exibe a matriz de confusão bruta.
print('Matriz de confusão (linhas = real, colunas = predito):')
print(cm)
# Calcula acurácia geral.
acuracia = accuracy_score(df_resultados['target_real'], df_resultados['target_predito'])
# Calcula F1 macro (média das duas classes).
f1_macro = f1_score(df_resultados['target_real'], df_resultados['target_predito'], average='macro')
# Exibe as métricas resumidas.
print(f'\nAcurácia: {acuracia:.4f}')
print(f'F1 macro: {f1_macro:.4f}')
# Exibe o relatório de classificação completo (precisão, recall, F1 por classe).
print('\nRelatório de classificação:')
print(classification_report(
    df_resultados['target_real'],
    df_resultados['target_predito'],
    target_names=['Furto', 'Roubo']
))

Matriz de confusão (linhas = real, colunas = predito):
[[48846 24071]
 [11580 27695]]

Acurácia: 0.6822
F1 macro: 0.6705

Relatório de classificação:
              precision    recall  f1-score   support

       Furto       0.81      0.67      0.73     72917
       Roubo       0.54      0.71      0.61     39275

    accuracy                           0.68    112192
   macro avg       0.67      0.69      0.67    112192
weighted avg       0.71      0.68      0.69    112192



In [13]:
# Importa matplotlib e configura backend Agg (sem interface gráfica, necessário para execução como script).
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# Importa seaborn para heatmap da matriz de confusão.
import seaborn as sns
# Cria uma figura com dois subplots lado a lado.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# --- Gráfico 1: Matriz de confusão como heatmap ---
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=ax1,
    cbar=False,
    xticklabels=['Furto (predito)', 'Roubo (predito)'],
    yticklabels=['Furto (real)', 'Roubo (real)']
)
# Define o título do gráfico da matriz de confusão.
ax1.set_title('Matriz de Confusão - Teste Final 2026', fontweight='bold')
# Define o rótulo do eixo Y.
ax1.set_ylabel('Classe Real')
# Define o rótulo do eixo X.
ax1.set_xlabel('Classe Predita')
# --- Gráfico 2: Comparação de distribuição Real vs Predito ---
# Conta as ocorrências por classe real.
contagem_real = df_resultados['classe_real'].value_counts().reindex(['Furto', 'Roubo']).fillna(0)
# Conta as ocorrências por classe predita.
contagem_predita = df_resultados['classe_predita'].value_counts().reindex(['Furto', 'Roubo']).fillna(0)
# Define as posições das barras no eixo X.
x = np.arange(2)
largura = 0.35
# Plota as barras da contagem real.
ax2.bar(x - largura/2, contagem_real.values, largura, label='Real', color='#4cc9f0')
# Plota as barras da contagem predita.
ax2.bar(x + largura/2, contagem_predita.values, largura, label='Predito', color='#ff2a3c')
# Define os rótulos do eixo X.
ax2.set_xticks(x)
ax2.set_xticklabels(['Furto', 'Roubo'])
# Define o título do gráfico de barras.
ax2.set_title('Distribuição de Classes: Real vs Predito (2026)', fontweight='bold')
# Define o rótulo do eixo Y.
ax2.set_ylabel('Quantidade de Ocorrências')
# Exibe a legenda.
ax2.legend()
# Adiciona os valores sobre as barras.
for i, v in enumerate(contagem_real.values):
    ax2.text(i - largura/2, v + max(contagem_real.values)*0.01, f'{int(v)}', ha='center', va='bottom')
for i, v in enumerate(contagem_predita.values):
    ax2.text(i + largura/2, v + max(contagem_predita.values)*0.01, f'{int(v)}', ha='center', va='bottom')
# Ajusta o espaçamento entre os subplots.
plt.tight_layout()
# Salva a figura como PNG (sem exibição interativa).
caminho_figura = RAIZ / 'reports' / 'images' / 'resultado_teste_final_2026.png'
caminho_figura.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(caminho_figura, dpi=150, bbox_inches='tight')
print(f'Figura salva em: {caminho_figura}')

Figura salva em: d:\Seguranca_Publica.InteligenteSP\reports\images\resultado_teste_final_2026.png


In [14]:
# Cria uma tabela comparativa entre classes reais e preditas.
tabela_comparativa = pd.DataFrame({
    'Classe': ['Furto', 'Roubo'],
    'Real': contagem_real.values.astype(int),
    'Predito': contagem_predita.values.astype(int)
})
# Calcula a diferença absoluta entre real e predito.
tabela_comparativa['Diferença'] = tabela_comparativa['Predito'] - tabela_comparativa['Real']
# Exibe a tabela comparativa.
print(tabela_comparativa.to_string())
# Exibe a porcentagem de acerto por classe.
print('\nPercentual de acerto por classe:')
for classe_nome, classe_cod in [('Furto', 0), ('Roubo', 1)]:
    # Seleciona as linhas da classe real.
    linhas_classe = df_resultados[df_resultados['target_real'] == classe_cod]
    # Conta quantas foram preditas corretamente.
    acertos = (linhas_classe['target_predito'] == classe_cod).sum()
    total = len(linhas_classe)
    percentual = (acertos / total * 100) if total > 0 else 0
    print(f'{classe_nome}: {acertos}/{total} = {percentual:.2f}% de acerto')
# Salva os resultados em arquivo CSV para auditoria.
caminho_saida = RAIZ / 'reports' / 'resultado_teste_final_2026.csv'
caminho_saida.parent.mkdir(parents=True, exist_ok=True)
df_resultados.to_csv(caminho_saida, index=False, encoding='utf-8')
print(f'\nResultados salvos em: {caminho_saida}')

  Classe   Real  Predito  Diferença
0  Furto  72917    60426     -12491
1  Roubo  39275    51766      12491

Percentual de acerto por classe:
Furto: 48846/72917 = 66.99% de acerto
Roubo: 27695/39275 = 70.52% de acerto

Resultados salvos em: d:\Seguranca_Publica.InteligenteSP\reports\resultado_teste_final_2026.csv
